# Graph2Edits 教程 1：环境配置

## 概述

本 notebook 对应整个 Graph2Edits 教学系列的第一步：把原始仓库中依赖 `Python 3.7 + PyTorch 1.6 + RDKit 2019.09` 的旧环境，整理成适合当前教学演示的现代 **CPU-only** 环境。

这个 notebook 的目标不是“把仓库黑盒装起来”，而是先把后续两份 notebook 需要的最小运行条件说明清楚，并且显式映射回原仓库的依赖假设。

### 本 notebook 内容

| 章节 | 内容 |
|------|------|
| 1 | 定位项目根目录与教程相关路径 |
| 2 | 对比原仓库环境与教学环境 |
| 3 | 生成 CPU-only 安装命令 |
| 4 | 验证源码导入与最小 smoke test |
| 5 | 说明现代 PyTorch / Jupyter 下的兼容性点 |

> 注意
>
> 本系列所有路径都从 `Chemical_Synthesis` 项目根目录出发计算，而不是从 notebook 所在目录出发写死绝对路径。

## 源码对应

- 文件：`source_repos/Graph2Edits/README.md`
- 文件：`source_repos/Graph2Edits/preprocess.py`
- 文件：`source_repos/Graph2Edits/prepare_data.py`
- 说明：原仓库 README 给的是 2022 年左右的旧环境；教学版保留代码主线，但升级到现代 CPU 环境并补上 notebook 友好的路径管理。

In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "source_repos").exists() and (candidate / "teaching_demos").exists():
            return candidate
    raise RuntimeError("无法定位项目根目录，请从 Chemical_Synthesis 项目内启动 notebook。")


PROJECT_ROOT = find_project_root(Path.cwd())
TUTORIAL_REL = Path("teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits")
REPO_REL = Path("source_repos/Graph2Edits")
ENV_REL = Path("envs/graph2edits_tutorial_envs")
DEMO_DATA_REL = TUTORIAL_REL / "data/demo_reactions.csv"

TUTORIAL_ROOT = PROJECT_ROOT / TUTORIAL_REL
REPO_ROOT = PROJECT_ROOT / REPO_REL
ENV_ROOT = PROJECT_ROOT / ENV_REL
DEMO_DATA_FILE = PROJECT_ROOT / DEMO_DATA_REL

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

path_table = {
    "PROJECT_ROOT": PROJECT_ROOT,
    "REPO_REL": REPO_REL,
    "TUTORIAL_REL": TUTORIAL_REL,
    "ENV_REL": ENV_REL,
    "DEMO_DATA_REL": DEMO_DATA_REL,
}

for key, value in path_table.items():
    print(f"{key}: {value}")

PROJECT_ROOT: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
REPO_REL: source_repos/Graph2Edits
TUTORIAL_REL: teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits
ENV_REL: envs/graph2edits_tutorial_envs
DEMO_DATA_REL: teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.2.graph2edits/data/demo_reactions.csv


## Step 1: 环境假设对比

### 为什么需要这一步？

如果直接照着原仓库 README 安装，很容易默认走到 `Python 3.7 + torch 1.6 + rdkit 2019.09` 的旧组合；这在今天已经不适合作为教学环境。

教学版要做的不是盲目追新，而是把依赖升级到现代版本，同时确保 `preprocess.py`、`prepare_data.py`、`models/graph2edits.py` 这些核心代码还能按原本思路运行。

### 原仓库与教学环境的差异

| 项目 | 原仓库 README | 教学环境 |
|------|---------------|----------|
| Python | 3.7 | 3.10+ |
| PyTorch | 1.6.0 | CPU-only `torch` |
| RDKit | 2019.09.2 | 现代 `rdkit` 轮子 |
| 运行入口 | 命令行脚本为主 | notebook + 命令行都可用 |
| 路径策略 | 默认从仓库根目录 `data/...` 读取 | 所有路径先定位项目根目录，再拼相对路径 |
| 训练设备 | 默认考虑 CUDA | 本教程只要求 CPU |

### 源码对应

- 文件：`source_repos/Graph2Edits/README.md`
- 函数：`train.build_model_config()`
- 说明：模型本体并不依赖 GPU 专用算子，因此教学演示完全可以用 CPU 版 `torch`。

In [2]:
import importlib
import platform
import pandas as pd

module_names = ["torch", "rdkit", "pandas", "joblib", "numpy", "matplotlib", "ipykernel"]
rows = []

for name in module_names:
    try:
        module = importlib.import_module(name)
        version = getattr(module, "__version__", "unknown")
        rows.append({"module": name, "status": "ok", "version": version})
    except Exception as exc:
        rows.append({"module": name, "status": type(exc).__name__, "version": str(exc)})

print(f"Python executable: {sys.executable}")
print(f"Python version: {platform.python_version()}")
display(pd.DataFrame(rows))

Python executable: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/graphretro_tutorial_envs/bin/python
Python version: 3.10.12


,module,status,version
0,torch,ok,2.1.0+cpu
1,rdkit,ok,2025.09.6
2,pandas,ok,2.3.3
3,joblib,ok,1.5.3
4,numpy,ok,1.26.4
5,matplotlib,ok,3.10.8
6,ipykernel,ok,7.2.0


## Step 2: 生成 CPU-only 安装命令

### 为什么需要这一步？

Graph2Edits 的教学目标是理解 edit 序列、图状态展开和模型打分逻辑，不是做大规模 GPU 训练。因此环境里只需要安装 **CPU 版 `torch`**。

### 核心思想

1. 先用项目根目录下的相对路径创建独立环境。
2. 常规科学计算依赖从 PyPI 安装。
3. `torch` 单独从 CPU wheel 源安装，避免误装 CUDA 版本。
4. 最后注册 Jupyter kernel，保证 notebook 真正运行在教学环境里。

### 现代环境下需要额外注意的兼容性

| 兼容点 | 原仓库状态 | 教学版处理 |
|--------|------------|------------|
| `torch.load` 默认行为 | 老版本无需考虑 | 新版本若加载 checkpoint，建议显式写 `weights_only=False` |
| notebook kernel | README 未覆盖 | 显式注册 `ipykernel` |
| 路径管理 | `data/...` 写死在仓库内部 | 先找项目根目录，再拼相对路径 |
| `torchvision` | README 给出 | 教学版不需要 |
| 本机 `python3` | 可能指向有问题的 `miniconda` 解释器 | 教学版显式使用 `/usr/bin/python3` 创建 venv |

> 说明
>
> 本教程不会修改 `Graph2Edits` 源码仓库里的依赖声明，而是在 notebook 中给出一个面向教学的现代环境方案。

In [3]:
install_commands = [
    f"cd {PROJECT_ROOT}",
    "/usr/bin/python3 -m venv envs/graph2edits_tutorial_envs",
    "source envs/graph2edits_tutorial_envs/bin/activate",
    "python -m pip install --upgrade pip",
    "python -m pip install rdkit pandas joblib numpy matplotlib ipykernel nbformat nbclient",
    "python -m pip install torch --index-url https://download.pytorch.org/whl/cpu",
    "python -m ipykernel install --user --name graph2edits_tutorial_envs --display-name 'Python (graph2edits_tutorial_envs)'",
]

print("请从项目根目录运行以下命令：\n")
for idx, command in enumerate(install_commands, start=1):
    print(f"{idx}. {command}")

请从项目根目录运行以下命令：

1. cd /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
2. /usr/bin/python3 -m venv envs/graph2edits_tutorial_envs
3. source envs/graph2edits_tutorial_envs/bin/activate
4. python -m pip install --upgrade pip
5. python -m pip install rdkit pandas joblib numpy matplotlib ipykernel nbformat nbclient
6. python -m pip install torch --index-url https://download.pytorch.org/whl/cpu
7. python -m ipykernel install --user --name graph2edits_tutorial_envs --display-name 'Python (graph2edits_tutorial_envs)'


## Step 3: 进行最小源码 smoke test

### 为什么需要这一步？

环境能安装成功，不等于 Graph2Edits 的源码一定能顺利导入。这里要验证三件事：

1. 能否读到教程里准备的 demo 反应数据。
2. 能否导入 `generate_reaction_edits`、`MolGraph`、`get_batch_graphs` 等后续 notebook 依赖的核心函数。
3. 能否把一个产物 SMILES 转成图张量。

### 源码对应

- 文件：`source_repos/Graph2Edits/canonicalize_prod.py`
- 文件：`source_repos/Graph2Edits/utils/generate_edits.py`
- 文件：`source_repos/Graph2Edits/utils/rxn_graphs.py`
- 文件：`source_repos/Graph2Edits/utils/collate_fn.py`

In [4]:
import pandas as pd
from rdkit import Chem

from utils.generate_edits import generate_reaction_edits
from utils.rxn_graphs import MolGraph
from utils.collate_fn import get_batch_graphs


demo_df = pd.read_csv(DEMO_DATA_FILE)
sample = demo_df.iloc[0]
rxn_data = generate_reaction_edits(
    sample["canonicalized_reaction"],
    kekulize=True,
    rxn_class=int(sample["class"]) - 1,
    rxn_id=sample["id"],
)

product_smi = rxn_data.rxn_smi.split(">>")[-1]
product_mol = Chem.MolFromSmiles(product_smi)
product_graph = MolGraph(product_mol)
product_tensors, product_scopes = get_batch_graphs([product_graph], use_rxn_class=False)

print("sample id:", sample["id"])
print("product smiles:", product_smi)
print("num edits:", len(rxn_data.edits))
print("atom tensor shape:", tuple(product_tensors[0].shape))
print("bond tensor shape:", tuple(product_tensors[1].shape))
print("atom scopes:", product_scopes[0])
print("bond scopes:", product_scopes[1])

sample id: US05849732
product smiles: [CH3:1][O:2][C:3](=[O:4])[C@H:5]([CH2:6][CH2:7][CH2:8][CH2:9][NH2:10])[NH:11][C:12](=[O:13])[NH:14][c:15]1[cH:16][c:17]([O:18][CH3:19])[cH:20][c:21]([C:22]([CH3:23])([CH3:24])[CH3:25])[c:26]1[OH:27]
num edits: 3
atom tensor shape: (28, 85)
bond tensor shape: (55, 97)
atom scopes: [(1, 27)]
bond scopes: [(1, 27)]


In [5]:
import torch


def torch_load_compat(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


print("torch version:", torch.__version__)
print("torch_load_compat(path) 已就绪，可用于后续 notebook 加载 batch 或 checkpoint。")

torch version: 2.1.0+cpu
torch_load_compat(path) 已就绪，可用于后续 notebook 加载 batch 或 checkpoint。


## 小结

这份 notebook 完成了三件事：

1. 明确了 Graph2Edits 教学系列的统一路径策略。
2. 给出了一个面向现代 CPU 环境的安装方案。
3. 验证了后续 notebook 需要的核心模块已经可以被导入并生成最小图张量。

下一份 notebook `2_数据处理.ipynb` 会从原始反应出发，逐步演示 Graph2Edits 如何把一条反应转换成 edit 序列、词表、图状态序列和训练 batch。